In [37]:
import pymysql
import pandas as pd
from typing import Optional
from DATA.stock_invest_function import get_db_host

def inspect_indicators_for_ticker(db_info,
                                  ticker,
                                  table_name="korea_fs_data_from_DART"):
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", 3306),
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )

    try:
        sql = f"""
            SELECT DISTINCT indicator
            FROM {table_name}
            WHERE ticker = %s
              AND indicator IS NOT NULL
              AND indicator <> ''
              AND indicator <> 'indicator'
            ORDER BY indicator
        """
        df = pd.read_sql(sql, conn, params=[ticker])
    finally:
        conn.close()

    print(f"▶ ticker={ticker} 에서 발견된 indicator 목록:")
    print(df["indicator"].tolist())
    return df["indicator"].tolist()


def get_indicator_from_db(db_info,
                          ticker,
                          indicator,
                          table_name="korea_fs_data_from_DART",
                          start_date=None,
                          end_date=None):

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", 3306),
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )

    try:
        sql = f"""
            SELECT
                date,
                company_name,
                ticker,
                indicator,
                value
            FROM {table_name}
            WHERE ticker = %s
              AND indicator = %s
              -- ✅ 1차: date 컬럼이 'date'인 가짜 행 제거
              AND date <> 'date'
        """
        params = [ticker, indicator]

        if start_date is not None:
            sql += " AND date >= %s"
            params.append(start_date)

        if end_date is not None:
            sql += " AND date <= %s"
            params.append(end_date)

        sql += " ORDER BY date"

        df = pd.read_sql(sql, conn, params=params)

    finally:
        conn.close()

    if df.empty:
        return df

    # ✅ 2차: 혹시 남아있을지 모르는 완전-헤더 행 제거
    mask_header = (
        (df["date"] == "date") &
        (df["company_name"] == "company_name") &
        (df["ticker"] == "ticker") &
        (df["indicator"] == "indicator") &
        (df["value"] == "value")
    )
    df = df[~mask_header].copy()

    # 타입 정리 (선택)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["value"] = (
        df["value"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

    df = df.sort_values("date").reset_index(drop=True)
    return df

In [38]:
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user' : 'stox7412',
    'password' : 'Apt106503!~',
    'database': 'investar'
}

df_cash = get_indicator_from_db(
    db_info=db_info,
    ticker="005680",
    indicator="Cash",
    table_name="korea_fs_data_from_DART",
)

df_cash.head()


,date,company_name,ticker,indicator,value


In [39]:
ind_list_001820 = inspect_indicators_for_ticker(db_info, "005680")

▶ ticker=005680 에서 발견된 indicator 목록:
['indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator', 'indicator']


In [40]:
def sample_trade_payables(db_info, table_name="korea_fs_data_from_DART"):
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", 3306),
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )

    try:
        sql = f"""
            SELECT date, company_name, ticker, indicator, value
            FROM {table_name}
            WHERE indicator = 'Trade_Payables'
            ORDER BY ticker, date
            LIMIT 50
        """
        df = pd.read_sql(sql, conn)
    finally:
        conn.close()

    return df

df_tp = sample_trade_payables(db_info)
df_tp.head()

,date,company_name,ticker,indicator,value
0,date,company_name,ticker,indicator,value
1,date,company_name,ticker,indicator,value
2,date,company_name,ticker,indicator,value
3,date,company_name,ticker,indicator,value
4,date,company_name,ticker,indicator,value


In [41]:
def list_all_indicators(db_info,
                        table_name="korea_fs_data_from_DART"):
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", 3306),
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )

    try:
        sql = f"""
            SELECT DISTINCT indicator
            FROM {table_name}
            WHERE indicator IS NOT NULL
              AND indicator <> ''
              AND indicator <> 'indicator'
        """
        df = pd.read_sql(sql, conn)
    finally:
        conn.close()

    indicators = sorted(df["indicator"].dropna().unique().tolist())
    print(f"📊 총 indicator 개수: {len(indicators)}")
    print(f"✅ ROA 존재 여부: {'ROA' in indicators}")
    print(f"✅ ROE 존재 여부: {'ROE' in indicators}")
    return indicators

all_indicators = list_all_indicators(db_info)

📊 총 indicator 개수: 1
✅ ROA 존재 여부: False
✅ ROE 존재 여부: False


In [42]:
all_indicators = list_all_indicators(db_info)
"ROA" in all_indicators, "ROE" in all_indicators

📊 총 indicator 개수: 1
✅ ROA 존재 여부: False
✅ ROE 존재 여부: False


(False, False)